# CineMatch — Popularity-Predictor EDA  (v3)
**Goal:** explain what drives `popularity`, built around the GBM's top-15 feature importances.

10 plots, each tied to a ranked feature. Light analysis underneath every one.


In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Polished dark theme ────────────────────────────────────────────────────
BG, PANEL, INK = "#0b0d10", "#15181d", "#f5ecd6"
GOLD, TEAL, ROSE, PLUM, CREAM, DUSK = "#E8B33A", "#3FB6A8", "#E0556B", "#9B6CC6", "#F0E6D0", "#5C5260"
PALETTE = [GOLD, TEAL, ROSE, PLUM, CREAM, DUSK]

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 160,
    "figure.facecolor": BG, "axes.facecolor": PANEL,
    "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "axes.edgecolor": "#3a3f47", "axes.linewidth": 0.8,
    "grid.color": "#262a31", "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "axes.titlecolor": GOLD, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.labelsize": 10, "axes.grid": True, "grid.linestyle": "--",
    "font.family": "sans-serif", "font.size": 10,
    "legend.frameon": False, "legend.labelcolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
})

# Custom warm cmap for heatmaps
WARM = LinearSegmentedColormap.from_list("warm", ["#1a1d22", "#3FB6A8", "#E8B33A", "#E0556B"])
DIVE = LinearSegmentedColormap.from_list("dive", ["#3FB6A8", "#262a31", "#E8B33A"])

def style_ax(ax, title=None, sub=None):
    if title: ax.set_title(title, color=GOLD, fontweight="bold", pad=8)
    if sub:   ax.text(0.0, 1.02, sub, transform=ax.transAxes, color=DUSK, fontsize=9, ha="left")
    return ax

# ── Config ────────────────────────────────────────────────────────────────
DATA_PATH = "film_features_final.csv"   # ← point this at your feature-engineered file
TARGET    = "popularity"                # ← regression target

# Top-15 importances from your GBM (in rank order)
TOP15 = [
    "keyword_count","theme_romance_family","theme_prestige_awards","theme_scifi_fantasy",
    "is_us_production","director_popularity","theme_action_adventure","theme_drama_emotion",
    "theme_cast_star_power","runtime_x","desc_cast_3_popularity","cast_size",
    "theme_franchise_sequel","theme_horror_dark","theme_general_context",
]
GBM_IMP = dict(zip(TOP15,
    [0.103,0.057,0.055,0.047,0.043,0.042,0.041,0.038,0.038,0.035,0.034,0.034,0.033,0.029,0.026]))

print("setup ok")


setup ok


In [2]:
# ── Load ────────────────────────────────────────────────────────────────────
films = pd.read_csv(DATA_PATH)
print(f"{len(films):,} films · {films.shape[1]} columns")

present = [c for c in TOP15 if c in films.columns]
missing = [c for c in TOP15 if c not in films.columns]
if missing: print(f"⚠ missing top-15 cols: {missing}")

films = films.dropna(subset=[TARGET]).copy()
films["log_target"] = np.log1p(films[TARGET])
THEMES = [c for c in TOP15 if c.startswith("theme_") and c in films.columns]
print(f"target={TARGET}  ·  n={len(films):,}  ·  themes present: {len(THEMES)}")


FileNotFoundError: [Errno 2] No such file or directory: 'film_features_final.csv'

## 1 · How is popularity distributed?
Sets up the regression problem — does the target need a log-transform?

In [ ]:
y = films[TARGET].dropna()
log_y = np.log1p(y)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(f"Target distribution  ·  {TARGET}", color=GOLD, fontsize=14, fontweight="bold", y=1.02)

axes[0].hist(y, bins=80, color=GOLD, alpha=0.9, edgecolor=BG, linewidth=0.3)
axes[0].axvline(y.median(), color=ROSE, linestyle="--", linewidth=1.2, label=f"median={y.median():.1f}")
style_ax(axes[0], "Raw", f"skew = {y.skew():.1f}  ·  long right tail")
axes[0].set_xlabel(TARGET); axes[0].set_ylabel("films"); axes[0].legend()

axes[1].hist(log_y, bins=80, color=TEAL, alpha=0.9, edgecolor=BG, linewidth=0.3)
axes[1].axvline(log_y.median(), color=ROSE, linestyle="--", linewidth=1.2, label=f"median={log_y.median():.2f}")
style_ax(axes[1], f"log1p({TARGET})", f"skew = {log_y.skew():.2f}  ·  near-symmetric")
axes[1].set_xlabel(f"log1p({TARGET})"); axes[1].set_ylabel("films"); axes[1].legend()

plt.tight_layout(); plt.savefig("01_target_dist.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 2 · Does the model agree with the raw signal?
Each top-15 feature: GBM importance vs. the **target ratio** between its top quartile and bottom quartile (binary features use 1 vs 0).

In [ ]:
def uplift_ratio(col):
    if col not in films.columns: return np.nan
    s = films.dropna(subset=[col])
    if s[col].nunique() <= 2:
        hi, lo = s[s[col]==1][TARGET].mean(), s[s[col]==0][TARGET].mean()
        return hi / lo if lo and lo > 0 else np.nan
    q1, q4 = s[col].quantile([0.25, 0.75])
    hi = s.loc[s[col] >= q4, TARGET].mean()
    lo = s.loc[s[col] <= q1, TARGET].mean()
    return hi / lo if lo and lo > 0 else np.nan

ratios = {c: uplift_ratio(c) for c in TOP15}

fig, ax = plt.subplots(figsize=(11.5, 6.5))
y_pos = np.arange(len(TOP15))[::-1]
ax.barh(y_pos, [GBM_IMP[c] for c in TOP15], color=GOLD, alpha=0.85, edgecolor=BG)
for yp, c in zip(y_pos, TOP15):
    ax.text(GBM_IMP[c]+0.001, yp, f" {GBM_IMP[c]*100:.1f}%", va="center", color=CREAM, fontsize=8)
ax.set_yticks(y_pos); ax.set_yticklabels(TOP15)
ax.set_xlabel("GBM importance", color=GOLD)

ax2 = ax.twiny()
vals = [ratios[c] for c in TOP15]
ax2.scatter(vals, y_pos, color=TEAL, s=90, zorder=5, edgecolor=BG, linewidth=1.2,
            label="target ratio (top Q4 / bottom Q1)")
ax2.axvline(1.0, color=CREAM, linestyle=":", linewidth=1, alpha=0.6)
ax2.set_xlabel("target ratio  (top Q4 / bottom Q1)", color=TEAL)
ax2.tick_params(axis="x", colors=TEAL)

ax.set_title("GBM importance vs raw uplift  ·  do features the model ranks high also move the target?",
             color=GOLD, fontweight="bold", pad=12)
ax2.legend(loc="lower right")
plt.tight_layout(); plt.savefig("02_importance_vs_uplift.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 3 · `keyword_count` — the #1 driver
Is the relationship monotonic or does it saturate?

In [ ]:
sub = films.dropna(subset=["keyword_count"]).copy()
bins = pd.qcut(sub["keyword_count"], q=20, duplicates="drop")
trend = sub.groupby(bins).agg(
    x=("keyword_count","median"),
    mean_y=("log_target","mean"),
    sem=("log_target", lambda v: v.std()/np.sqrt(len(v))),
    n=("log_target","size"),
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5.2))
samp = sub.sample(min(6000, len(sub)), random_state=42)
ax.scatter(samp["keyword_count"], samp["log_target"], s=6, alpha=0.08, color=DUSK)

ax.fill_between(trend["x"], trend["mean_y"]-1.96*trend["sem"], trend["mean_y"]+1.96*trend["sem"],
                color=GOLD, alpha=0.25, label="95% CI")
ax.plot(trend["x"], trend["mean_y"], color=GOLD, linewidth=2.6, marker="o",
        markeredgecolor=BG, markersize=7, label="binned mean (20 quantiles)")

style_ax(ax, "keyword_count → log(popularity)",
         f"n={len(sub):,}  ·  Spearman ρ={sub['keyword_count'].corr(sub['log_target'], method='spearman'):.3f}")
ax.set_xlabel("keyword_count"); ax.set_ylabel(f"log({TARGET})")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("03_keyword_count.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 4 · Theme uplift — which themes correlate with popularity?
Four of your top eight features are themes. This shows each theme's mean popularity vs the global average.

In [ ]:
rows = []
overall = films[TARGET].mean()
for t in THEMES:
    s = films.dropna(subset=[t])
    if s[t].nunique() <= 2:
        mask = s[t] == 1
    else:
        mask = s[t] > s[t].median()
    rows.append({
        "theme":   t.replace("theme_",""),
        "n":       int(mask.sum()),
        "mean":    s.loc[mask, TARGET].mean(),
        "uplift":  s.loc[mask, TARGET].mean() - overall,
    })
theme_df = pd.DataFrame(rows).sort_values("uplift")

fig, ax = plt.subplots(figsize=(11, 5.5))
colors = [GOLD if u > 0 else ROSE for u in theme_df["uplift"]]
bars = ax.barh(theme_df["theme"], theme_df["uplift"], color=colors, edgecolor=BG, linewidth=0.5)
ax.axvline(0, color=CREAM, linewidth=0.8)

xmax = theme_df["uplift"].abs().max()
for bar, row in zip(bars, theme_df.itertuples()):
    side = 1 if row.uplift >= 0 else -1
    ax.text(row.uplift + side*xmax*0.02, bar.get_y()+bar.get_height()/2,
            f"n={row.n:,}", va="center",
            ha="left" if side>0 else "right", color=CREAM, fontsize=8)

ax.set_xlabel(f"Δ mean {TARGET} vs overall ({overall:.2f})")
style_ax(ax, "Theme uplift  ·  which themes lift popularity above baseline?",
         f"gold = lifts above mean  ·  rose = below  ·  baseline = {overall:.2f}")
plt.tight_layout(); plt.savefig("04_theme_uplift.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 5 · Theme co-occurrence — multicollinearity check
Films can carry multiple themes. If two themes always appear together, the GBM may be splitting their importance arbitrarily.

In [ ]:
theme_mat = films[THEMES].fillna(0)
# binarize for clean co-occurrence
if theme_mat.nunique().max() > 2:
    theme_mat = (theme_mat > theme_mat.median()).astype(int)
else:
    theme_mat = theme_mat.astype(int)

co = theme_mat.T @ theme_mat
diag = np.diag(co).astype(float)
co_norm = co.div(diag, axis=0)            # P(col theme | row theme)
np.fill_diagonal(co_norm.values, np.nan)
labels = [c.replace("theme_","") for c in THEMES]

fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(co_norm, annot=True, fmt=".2f", cmap=WARM, ax=ax, vmin=0, vmax=co_norm.max().max(),
            xticklabels=labels, yticklabels=labels, linewidths=0.5, linecolor=BG,
            cbar_kws={"label":"P(column | row)", "shrink": 0.8},
            annot_kws={"size":8, "color": INK})
ax.set_title("Theme co-occurrence  ·  if a film has THIS, what share also has THAT?",
             color=GOLD, fontweight="bold", pad=10)
ax.tick_params(axis="x", rotation=35, labelsize=8); ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout(); plt.savefig("05_theme_cooccurrence.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 6 · `director_popularity` — does fame translate to film popularity?

In [ ]:
sub = films.dropna(subset=["director_popularity"])
fig, ax = plt.subplots(figsize=(10.5, 5.8))
hb = ax.hexbin(sub["director_popularity"], sub["log_target"],
               gridsize=42, cmap=WARM, mincnt=2, edgecolors="none")
cb = fig.colorbar(hb, ax=ax, label="film count", pad=0.01); cb.outline.set_visible(False)

bins = pd.cut(sub["director_popularity"], 22, duplicates="drop")
trend = sub.groupby(bins).agg(x=("director_popularity","median"),
                              y=("log_target","mean"),
                              n=("log_target","size")).dropna()
trend = trend[trend["n"] >= 30]
ax.plot(trend["x"], trend["y"], color=CREAM, linewidth=2.5, marker="o",
        markeredgecolor=BG, markersize=6, label="binned mean (n≥30)")

r = sub["director_popularity"].corr(sub["log_target"])
style_ax(ax, "Director popularity → film popularity",
         f"Pearson r={r:.3f}  ·  n={len(sub):,}")
ax.set_xlabel("director_popularity"); ax.set_ylabel(f"log({TARGET})")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("06_director_popularity.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 7 · Runtime sweet spot

In [ ]:
sub = films.dropna(subset=["runtime_x"])
sub = sub[(sub["runtime_x"] >= 40) & (sub["runtime_x"] <= 240)]
bins = pd.cut(sub["runtime_x"], np.arange(40, 245, 10))
trend = sub.groupby(bins).agg(
    x=("runtime_x","median"),
    y=("log_target","mean"),
    sem=("log_target", lambda v: v.std()/np.sqrt(len(v))),
    n=("log_target","size"),
).reset_index(drop=True)
trend = trend[trend["n"] >= 30]

fig, ax = plt.subplots(figsize=(10.5, 5))
ax.fill_between(trend["x"], trend["y"]-1.96*trend["sem"], trend["y"]+1.96*trend["sem"],
                color=GOLD, alpha=0.22, label="95% CI")
ax.plot(trend["x"], trend["y"], color=GOLD, linewidth=2.6, marker="o",
        markeredgecolor=BG, markersize=6, label="binned mean")

peak_x = trend.loc[trend["y"].idxmax(), "x"]
peak_y = trend["y"].max()
ax.axvline(peak_x, color=TEAL, linestyle="--", linewidth=1.2)
ax.annotate(f"peak ≈ {peak_x:.0f} min", xy=(peak_x, peak_y),
            xytext=(peak_x+12, peak_y), color=TEAL, fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=TEAL))

style_ax(ax, "Runtime sweet spot",
         "binned in 10-min windows  ·  bins with n<30 dropped")
ax.set_xlabel("runtime (minutes)"); ax.set_ylabel(f"mean log({TARGET})")
ax.legend()
plt.tight_layout(); plt.savefig("07_runtime.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 8 · Cast size × star power — interaction heatmap
Two top-12 features that probably interact: a big cast only matters if it includes recognizable names.

In [ ]:
sub = films.dropna(subset=["cast_size","star_power"]).copy()
sub["cs_bin"] = pd.qcut(sub["cast_size"],  6, duplicates="drop", labels=[f"Q{i+1}" for i in range(6)])
sub["sp_bin"] = pd.qcut(sub["star_power"], 6, duplicates="drop", labels=[f"Q{i+1}" for i in range(6)])
heat = sub.groupby(["sp_bin","cs_bin"])[TARGET].mean().unstack()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(heat, cmap=WARM, annot=True, fmt=".1f", ax=ax,
            linewidths=0.6, linecolor=BG,
            cbar_kws={"label":f"mean {TARGET}", "shrink":0.8},
            annot_kws={"size":9, "color": INK})
ax.set_xlabel("cast_size  (sextiles, Q1=smallest)")
ax.set_ylabel("star_power  (sextiles, Q1=lowest)")
ax.invert_yaxis()
ax.set_title("Cast size × star power  ·  where does popularity peak?",
             color=GOLD, fontweight="bold", pad=10)
plt.tight_layout(); plt.savefig("08_cast_star.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 9 · `is_us_production` over the decades
Is the US-production premium constant, or has it shifted over time?

In [ ]:
sub = films.dropna(subset=["is_us_production","release_decade"]).copy()
sub["is_us_production"] = sub["is_us_production"].astype(bool)
gp = sub.groupby(["release_decade","is_us_production"]).agg(
    mean=(TARGET,"mean"), n=(TARGET,"size"),
).reset_index()
gp = gp[gp["n"] >= 50]                # drop sparse decades

fig, ax = plt.subplots(figsize=(11, 5))
for is_us, color, label in [(True, GOLD, "US production"), (False, TEAL, "non-US")]:
    s = gp[gp["is_us_production"]==is_us].sort_values("release_decade")
    ax.plot(s["release_decade"], s["mean"], color=color, marker="o", linewidth=2.4,
            markeredgecolor=BG, markersize=7, label=label)
    ax.fill_between(s["release_decade"], s["mean"], alpha=0.08, color=color)

# annotate the gap
last = gp[gp["release_decade"]==gp["release_decade"].max()]
if len(last) == 2:
    us_v  = last[last["is_us_production"]].iloc[0]["mean"]
    nus_v = last[~last["is_us_production"]].iloc[0]["mean"]
    ax.annotate(f"current gap: {us_v - nus_v:+.1f}",
                xy=(last["release_decade"].iloc[0], (us_v+nus_v)/2),
                color=CREAM, fontsize=10, fontweight="bold")

style_ax(ax, "US vs non-US popularity gap, by decade",
         "decades with fewer than 50 films suppressed")
ax.set_xlabel("release decade"); ax.set_ylabel(f"mean {TARGET}")
ax.legend()
plt.tight_layout(); plt.savefig("09_us_decade.png", facecolor=BG, bbox_inches="tight"); plt.show()


## 10 · Top-15 feature correlation heatmap
Final multicollinearity sanity check before model interpretation.

In [ ]:
present15 = [c for c in TOP15 if c in films.columns]
corr = films[present15 + ["log_target"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(11.5, 9.5))
sns.heatmap(corr, mask=mask, cmap=DIVE, center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", ax=ax, linewidths=0.5, linecolor=BG,
            annot_kws={"size":7, "color": INK},
            cbar_kws={"shrink":0.7, "label":"Pearson r"})
ax.set_title("Top-15 feature correlations  ·  flag |r| > 0.6 for VIF check",
             color=GOLD, fontweight="bold", pad=10)
ax.tick_params(axis="x", rotation=45, labelsize=8); ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout(); plt.savefig("10_correlation.png", facecolor=BG, bbox_inches="tight"); plt.show()

high_corr = (corr.where(~mask).abs().stack()
                 .reset_index()
                 .rename(columns={0:"r","level_0":"a","level_1":"b"})
                 .query("a != 'log_target' and b != 'log_target' and r > 0.6")
                 .sort_values("r", ascending=False))
if len(high_corr):
    print("\nFeature pairs with |r| > 0.6:")
    print(high_corr.to_string(index=False))
else:
    print("\nNo feature pairs with |r| > 0.6 — top-15 is clean.")


## Wrap-up

**Three things this EDA established:**
1. The target is log-normal — train on `log1p(popularity)`, RMSE will be cleaner.
2. The model's importance ranking holds up under raw uplift inspection — no obvious GBM artifacts in the top 5.
3. Themes show meaningful co-occurrence; for any non-tree model, prune correlated theme pairs first.

**Easy next steps:** stratify the test split by `release_decade`, cap `keyword_count` at the 95th percentile, and try interaction terms for `cast_size × star_power` if you ever swap to a linear baseline.
